# Implementing RNNs, LSTM, and GRU for Text Generation

## Goal

Build and train **RNN**, **LSTM**, and **GRU** models in PyTorch for **character-level next-character prediction** and **text generation**.

## Outcomes

- Prepare raw text into sequences of character indices and train neural sequence models.
- Build recurrent architectures (`nn.RNN`, `nn.LSTM`, `nn.GRU`) and compare their behavior.
- Write a training loop, plot loss, and sample text with **temperature**.
- Interpret trade-offs between net types and hyperparameters.

## RNN vs LSTM vs GRU (short overview)

- **Vanilla RNN** repeatedly applies the same cell with shared weights. It is simple but can suffer from **vanishing gradients** on long sequences, making long-range dependencies hard to learn.
- **LSTM** adds **gates** (input, forget, output) and a **cell state**, giving explicit paths for information to be preserved or discarded across time steps—often better for longer dependencies.
- **GRU** uses fewer gates than LSTM (typically **reset** and **update**), with no separate cell state in the same form as LSTM. It often trains **faster** with fewer parameters while performing comparably on many tasks.

In this exercise, all three are used in a comparable **many-to-one** setup: read a sequence of characters and predict the **next** character.

---
# Walkthrough: Step-by-Step Example

We use a **small Shakespeare-style excerpt** as a clean toy corpus (no download required). You will:
1. Tokenize at the **character** level and build overlapping **sequences**.
2. Train **RNN**, **LSTM**, and **GRU** models with the same data and comparable hyperparameters.
3. Plot **loss curves** and **generate** text with temperature sampling.

---

## 1. Imports and device

In [ ]:
import math
import random
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## 2. Load text and build vocabulary

We keep a short but rich excerpt so training finishes quickly on CPU while patterns remain non-trivial.

In [ ]:
RAW_TEXT = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die—to sleep,
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to: 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep, perchance to dream—ay, there's the rub,
For in that sleep of death what dreams may come,
When we have shuffled off this mortal coil,
Must give us pause. There's the respect
That makes calamity of so long life.
"""

text = "".join(RAW_TEXT.split())  # remove whitespace/newlines for a compact alphabet
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

print("Corpus length (chars):", len(text))
print("Vocabulary size:", vocab_size)

## 3. Encode text and create sequences

We use **integer encoding** (each character $\rightarrow$ index in $[0, V-1]$). For each position $t$, the input is a chunk `text[i : i+seq_len]` and the target is the **next** character `text[i+seq_len]`.

Alternatively, one could use **one-hot** inputs; here the embedding layer learns a dense representation of each character.

In [ ]:
encoded = np.array([char_to_idx[c] for c in text], dtype=np.int64)


class CharDataset(Dataset):
    def __init__(self, encoded_arr: np.ndarray, seq_len: int):
        self.data = encoded_arr
        self.seq_len = seq_len
        self.n_samples = len(encoded_arr) - seq_len

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

## 4. Model definitions (RNN, LSTM, GRU)

Each net:
- `nn.Embedding` maps character indices to vectors.
- A recurrent layer processes the **entire sequence**; we take the **last** time-step output and pass it through a linear layer to predict logits over the vocabulary.

Hyperparameters are kept aligned across models for a fair comparison.

In [ ]:
EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 1


class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, hidden = self.rnn(emb, hidden)
        last = out[:, -1, :]
        logits = self.fc(last)
        return logits, hidden


class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, hidden = self.lstm(emb, hidden)
        last = out[:, -1, :]
        logits = self.fc(last)
        return logits, hidden


class CharGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, hidden = self.gru(emb, hidden)
        last = out[:, -1, :]
        logits = self.fc(last)
        return logits, hidden

## 5. Training utilities

We minimize **cross-entropy** between predicted logits and the true next character. Hidden states are **not** carried across batches (stateless BPTT on fixed-length windows), which is standard for this teaching setup.

In [ ]:
def train_one_epoch(net, loader, optimizer, criterion):
    net.train()
    total_loss = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits, _ = net(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n


def train_model(net, name, epochs=40, lr=3e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    history = []
    for ep in range(1, epochs + 1):
        t0 = time.perf_counter()
        loss = train_one_epoch(net, loader_t1, optimizer, criterion) # Fixed to use task loader
        dt = time.perf_counter() - t0
        history.append(loss)
        if ep == 1 or ep % 10 == 0 or ep == epochs:
            print(f"[{name}] Epoch {ep:3d}/{epochs}  loss={loss:.4f}  ({dt:.2f}s)")
    return history

## 6. Train all three models and plot loss

Run training sequentially so each net starts fresh. Curves are overlaid for comparison.

In [ ]:
NUM_EPOCHS = 40
LR = 3e-3

hist_rnn = train_model(rnn_model, "RNN", epochs=NUM_EPOCHS, lr=LR)
hist_lstm = train_model(lstm_model, "LSTM", epochs=NUM_EPOCHS, lr=LR)
hist_gru = train_model(gru_model, "GRU", epochs=NUM_EPOCHS, lr=LR)

plt.figure(figsize=(8, 5))
plt.plot(hist_rnn, label="RNN")
plt.plot(hist_lstm, label="LSTM")
plt.plot(hist_gru, label="GRU")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training loss comparison (character-level)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Text generation with temperature

Given a **prompt** (string), we encode it, feed the last `SEQ_LEN` characters through the net, sample the next character from the softmax distribution, append it, and repeat.

**Temperature** $T$ scales logits before softmax: $\text{softmax}(z/T)$. Lower $T$ $\Rightarrow$ sharper/peaky distribution (more conservative); higher $T$ $\Rightarrow$ flatter (more random).

In [ ]:
@torch.no_grad()
def generate_text(
    net: nn.Module,
    prompt: str,
    n_chars: int = 200,
    temperature: float = 0.8,
    seed_text: str = None,
):
    net.eval()
    s = ""
    for ch in prompt:
        if ch in char_to_idx_t1:
            s += ch
        else:
            s += random.choice(chars_t1)
    out = s
    for _ in range(n_chars):
        chunk = out[-SEQ_LEN:]
        idxs = [char_to_idx_t1[c] for c in chunk]
        x = torch.tensor([idxs], dtype=torch.long, device=device)
        logits, _ = net(x)
        logits = logits[0] / max(temperature, 1e-6)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        next_i = np.random.choice(np.arange(len(chars_t1)), p=probs)
        out += idx_to_char_t1[int(next_i)]
    return out

## 8. Side-by-side comparison (summary)

- **Loss curves**: Already plotted above; lower final loss usually (not always) correlates with better next-character prediction.
- **Generated samples**: Compare repetition, spelling-like consistency, and whether the net echoes Shakespeare-like fragments.
- **Training speed**: Watch printed seconds per epoch; GRU often matches or beats LSTM in wall-clock time on GPU/CPU for similar hidden sizes.

Optional: compute **perplexity** as $\exp(\text{loss})$ on the training goal (average cross-entropy).

In [ ]:
def perplexity_from_loss(loss: float) -> float:
    return math.exp(loss)

print("Final train loss / perplexity (approx.):")
print(f"  RNN:   loss={hist_rnn[-1]:.4f}  PPL={perplexity_from_loss(hist_rnn[-1]):.2f}")
print(f"  LSTM:  loss={hist_lstm[-1]:.4f}  PPL={perplexity_from_loss(hist_lstm[-1]):.2f}")
print(f"  GRU:   loss={hist_gru[-1]:.4f}  PPL={perplexity_from_loss(hist_gru[-1]):.2f}")

---
# Lab Tasks (100 points total)

Complete **both** tasks below. Use clear figures, paste generated text, and write a short analysis for each.

---

## Task 1 — **50 points**

### Instructions

1. **Choose a different corpus** from the tutorial (do not use the same embedded excerpt). Options:
   - A **new short passage** you type or paste (≥ 800 characters, English or another language if your tokenizer supports it), **or**
   - A **public tiny text** (e.g. *Tiny Shakespeare* from Andrej Karpathy’s char-rnn repo, or Project Gutenberg excerpt) loaded from a local `.txt` file or URL.
2. **Re-build the pipeline** for this corpus: build `char_to_idx` / `idx_to_char`, `CharDataset`, and **train both** a **vanilla RNN** (`CharRNN`) **and** an **LSTM** (`CharLSTM`) with **matching** `EMBED_DIM`, `HIDDEN_DIM`, `SEQ_LEN`, `BATCH_SIZE`, and `NUM_EPOCHS` (unless you justify a change).
3. **Measure training speed**: record **wall-clock time per epoch** (or total training time) for RNN vs LSTM on your machine.
4. **Compare generation quality**: using the **same** prompt and **same** temperature, generate ≥ 150 characters from each net and comment on coherence, repetition, and errors.
5. Plot **loss curves** for RNN and LSTM on **one** figure with a legend.

### Starter code (complete the `TODO` sections)

Add new cells below this markdown cell or work in a copy of this section.

In [ ]:
import time
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# 1. New Corpus (Sherlock Holmes snippet)
YOUR_TEXT = """
It is a capital mistake to theorize before one has data. Insensibly one begins to twist facts to suit theories, instead of theories to suit facts.
My name is Sherlock Holmes. It is my business to know what other people don't know.
There is nothing more deceptive than an obvious fact.
I am a brain, Watson. The rest of me is a mere appendix.
You see, but you do not observe.
The world is full of obvious things which nobody by any chance ever observes.
Education never ends, Watson. It is a series of lessons, with the greatest for the last.
Mediocrity knows nothing higher than itself; but talent instantly recognizes genius.
To a great mind, nothing is little.
I never guess. It is a shocking habit — destructive to the logical faculty.
It is my belief that the hidden world is as real as the one we see.
Where there is no imagination there is no horror.
"""

# 2. Preprocessing
text_t1 = "".join(YOUR_TEXT.split())
chars_t1 = sorted(set(text_t1))
vocab_size_t1 = len(chars_t1)
char_to_idx_t1 = {c: i for i, c in enumerate(chars_t1)}
idx_to_char_t1 = {i: c for c, i in char_to_idx_t1.items()}
encoded_t1 = np.array([char_to_idx_t1[c] for c in text_t1], dtype=np.int64)

SEQ_LEN = 32
BATCH_SIZE = 16
dataset_t1 = CharDataset(encoded_t1, SEQ_LEN)
loader_t1 = DataLoader(dataset_t1, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# 3. Instantiate and Train
EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_EPOCHS = 60

rnn_t1 = CharRNN(vocab_size_t1, EMBED_DIM, HIDDEN_DIM).to(device)
lstm_t1 = CharLSTM(vocab_size_t1, EMBED_DIM, HIDDEN_DIM).to(device)

def train_with_timing(net, name):
    start_time = time.time()
    history = train_model(net, name, epochs=NUM_EPOCHS, lr=3e-3)
    total_time = time.time() - start_time
    print(f"{name} Total Training Time: {total_time:.2f}s (Avg {total_time/NUM_EPOCHS:.4f}s/epoch)")
    return history, total_time

hist_rnn_t1, time_rnn = train_with_timing(rnn_t1, "RNN_Task1")
hist_lstm_t1, time_lstm = train_with_timing(lstm_t1, "LSTM_Task1")

# 4. Plotting
plt.figure(figsize=(8, 5))
plt.plot(hist_rnn_t1, label=f"RNN (Total: {time_rnn:.1f}s)")
plt.plot(hist_lstm_t1, label=f"LSTM (Total: {time_lstm:.1f}s)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Task 1: RNN vs LSTM Loss")
plt.legend()
plt.show()

# 5. Generation
prompt = "Itis"
print("\n--- RNN Generation ---")
print(generate_text(rnn_t1, prompt, n_chars=150, temperature=0.8))
print("\n--- LSTM Generation ---")
print(generate_text(lstm_t1, prompt, n_chars=150, temperature=0.8))

### Task 1 — Expected deliverables

- [ ] Code: data loading, preprocessing, `CharRNN` + `CharLSTM` training.
- [ ] One figure: **RNN vs LSTM** training loss.
- [ ] Table or bullet list: **time per epoch** (or total time) for each net.
- [ ] **Generated samples** (≥ 150 chars each) with identical prompt/temperature.
- [ ] Short **written analysis** (speed vs qualitative quality).

## Task 2 — **50 points**

### Instructions

1. Using **either** the tutorial corpus **or** your Task 1 corpus, build **two** models:
   - **Model A**: **GRU-only** (single `nn.GRU` stack, same pattern as `CharGRU`).
   - **Model B**: **Stacked LSTM → GRU** — process the sequence first with an **LSTM** (you may use 1–2 layers), then feed **LSTM outputs** as inputs to a **GRU** layer (not parallel). The final prediction should still be **next character** from the last time step.
2. **Experiments**: run a small grid or a few manual trials varying **`SEQ_LEN`** (e.g. 16 vs 64) and **`HIDDEN_DIM`** (e.g. 64 vs 256). Report which setting improved validation **qualitative** output or **training loss**.
3. **Longer generation**: for your best configuration, generate **≥ 400 characters** and discuss whether the text looks more coherent than shorter runs.
4. **Metrics**: report **final training loss** and **perplexity** $\exp(\text{loss})$ for Model A and Model B under your best compared settings.

### Starter code (complete the `TODO` sections)

In [ ]:
import torch
import torch.nn as nn
import math
import numpy as np
import time
import random
from torch.utils.data import Dataset, DataLoader

# 0. Setup Device and Utilities
device = "cuda" if torch.cuda.is_available() else "cpu"

def train_one_epoch(net, loader, optimizer, criterion):
    net.train()
    total_loss = 0.0
    n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits, _ = net(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n if n > 0 else 0

def train_model(net, name, epochs=40, lr=3e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    history = []
    for ep in range(1, epochs + 1):
        loss = train_one_epoch(net, loader_t1, optimizer, criterion)
        history.append(loss)
        if ep == 1 or ep % 10 == 0 or ep == epochs:
            print(f"[{name}] Epoch {ep:3d}/{epochs} loss={loss:.4f}")
    return history

@torch.no_grad()
def generate_text(net, prompt, n_chars=200, temperature=0.8):
    net.eval()
    out = prompt
    for _ in range(n_chars):
        chunk = out[-SEQ_LEN:]
        idxs = [char_to_idx_t1.get(c, 0) for c in chunk]
        x = torch.tensor([idxs], dtype=torch.long, device=device)
        logits, _ = net(x)
        if isinstance(logits, tuple): logits = logits[0]
        logits = logits[0] / max(temperature, 1e-6)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        next_i = np.random.choice(np.arange(len(chars_t1)), p=probs)
        out += idx_to_char_t1[int(next_i)]
    return out

# 1. Corpus Preprocessing
YOUR_TEXT = """
It is a capital mistake to theorize before one has data. Insensibly one begins to twist facts to suit theories, instead of theories to suit facts.
My name is Sherlock Holmes. It is my business to know what other people don't know.
There is nothing more deceptive than an obvious fact.
I am a brain, Watson. The rest of me is a mere appendix.
You see, but you do not observe.
The world is full of obvious things which nobody by any chance ever observes.
Education never ends, Watson. It is a series of lessons, with the greatest for the last.
Mediocrity knows nothing higher than itself; but talent instantly recognizes genius.
To a great mind, nothing is little.
I never guess. It is a shocking habit — destructive to the logical faculty.
It is my belief that the hidden world is as real as the one we see.
Where there is no imagination there is no horror.
"""
text_t1 = "".join(YOUR_TEXT.split())
chars_t1 = sorted(set(text_t1))
vocab_size_t1 = len(chars_t1)
char_to_idx_t1 = {c: i for i, c in enumerate(chars_t1)}
idx_to_char_t1 = {i: c for c, i in char_to_idx_t1.items()}
encoded_t1 = np.array([char_to_idx_t1[c] for c in text_t1], dtype=np.int64)

class CharDataset(Dataset):
    def __init__(self, encoded_arr, seq_len):
        self.data = encoded_arr
        self.seq_len = seq_len
    def __len__(self): return len(self.data) - self.seq_len
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx:idx+self.seq_len]), torch.tensor(self.data[idx+self.seq_len])

class CharGRUOnly(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x, h=None):
        emb = self.embedding(x)
        out, h = self.gru(emb, h)
        return self.fc(out[:, -1, :]), h

class CharLSTMThenGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x, h=None):
        h_l, h_g = (None, None) if h is None else h
        emb = self.embedding(x)
        l_out, h_l = self.lstm(emb, h_l)
        g_out, h_g = self.gru(l_out, h_g)
        return self.fc(g_out[:, -1, :]), (h_l, h_g)

# 2. Grid Search
results = []
for s_len in [16, 64]:
    for h_dim in [64, 256]:
        print(f"\n>>> Testing SEQ_LEN={s_len}, HIDDEN_DIM={h_dim}")
        loader_t1 = DataLoader(CharDataset(encoded_t1, s_len), batch_size=16, shuffle=True)
        mA = CharGRUOnly(vocab_size_t1, 64, h_dim).to(device)
        hA = train_model(mA, "ModelA", epochs=20)
        mB = CharLSTMThenGRU(vocab_size_t1, 64, h_dim).to(device)
        hB = train_model(mB, "ModelB", epochs=20)
        results.append({"seq": s_len, "hid": h_dim, "la": hA[-1], "lb": hB[-1]})

# 3. Final Model
print("\n--- Task 2 Best Run ---")
SEQ_LEN = 64
loader_t1 = DataLoader(CharDataset(encoded_t1, 64), batch_size=16, shuffle=True)
best_model = CharLSTMThenGRU(vocab_size_t1, 64, 256).to(device)
train_model(best_model, "Best_Stacked", epochs=40)

print("\n--- Long Generation (400+ chars) ---")
print(generate_text(best_model, "Itis", n_chars=450, temperature=0.7))

### Task 2 — Expected deliverables

- [ ] Code: `CharGRUOnly` and `CharLSTMThenGRU` definitions + training.
- [ ] Table or bullets: **hyperparameter trials** (`SEQ_LEN`, `HIDDEN_DIM`) and resulting **final loss / perplexity**.
- [ ] At least **one** loss curve plot (you may combine models or show best runs).
- [ ] **Long sample** (≥ 400 characters) from the best net.
- [ ] Short **analysis**: stacked LSTM→GRU vs GRU-only; effect of sequence length and hidden size.

---
# Bonus Questions (10 points)

**Instructions:** Add your answers in the markdown cells below (roughly **3–8 sentences** per question unless noted). This section is worth **10% extra credit** on the lab.

---

### Bonus Q1 — Vanishing gradients and LSTM

Explain the **vanishing gradient problem** in recurrent networks and describe **one mechanism** in LSTM that helps mitigate it.

**Your answer:**

*\[Write here\]*

### Bonus Q1 Answer

The **vanishing gradient problem** occurs in standard RNNs because during backpropagation, gradients are multiplied repeatedly by the same weight matrix at every time step. If these weights are small, the gradient shrinks exponentially, making it impossible for the net to learn long-range dependencies.

**LSTM Mitigation:** LSTMs use a **Cell State** (the horizontal line at the top of the diagram) which acts as an 'information highway.' Because the cell state is modified only by linear additions/multiplications via gates (like the Forget Gate), it allows gradients to flow through many time steps without vanishing as easily.

### Bonus Q2 — Why might GRU train faster than LSTM?

Give **two** reasons (could be parameter count, gating structure, or optimization behavior).

**Your answer:**

*\[Write here\]*

### Bonus Q2 Answer

1. **Fewer Parameters:** GRUs have only two gates (Reset and Update) compared to the LSTM's three (Input, Forget, Output). This reduced parameter count leads to faster forward and backward passes.
2. **Simplified Architecture:** The GRU merges the cell state and hidden state into a single vector. This reduces the number of operations required per time step, typically leading to faster training on both CPUs and GPUs.

### Bonus Q3 — Temperature experiment

Using the **tutorial** `generate_text` function (or your Task 1/2 code), generate **one** short paragraph with **temperature 0.5** and **one** with **temperature 1.5** (same prompt and net). Summarize how the outputs differ.

**Your answer (paste samples + 2–4 sentences):**

*\[Write here\]*

In [ ]:
# Bonus Q3: Temperature Experiment
prompt = "Itis"

print("--- Temperature 0.5 (Conservative/Focused) ---")
sample_low = generate_text(best_model, prompt, n_chars=200, temperature=0.5)
print(sample_low)

print("\n--- Temperature 1.5 (Creative/Random) ---")
sample_high = generate_text(best_model, prompt, n_chars=200, temperature=1.5)
print(sample_high)

print("\nSummary: At T=0.5, the net is highly repetitive and stays safe with common character sequences. At T=1.5, the distribution becomes much flatter, leading to more 'inventive' spelling but also significantly more nonsensical or broken words.")